# Requests to the Server

In [1]:
import pandas as pd
import requests
from sklearn.metrics import confusion_matrix

In [2]:
# # Local link
# post_url = 'http://localhost:9696/predict'

# Global link
post_url = 'http://card-fraud-prediction-serving-env.eba-fmgpcdqy.eu-north-1.elasticbeanstalk.com/predict'

## Data importing

In [3]:
DATA_PATH = '../data/'

df = pd.read_csv(DATA_PATH + 'train_1.csv')

## Requests

In [4]:
def server_request(person_data):
    if (type(person_data) == pd.DataFrame):
        person_data_dict = person_data.to_dict(orient='records')
    else:
        person_data_dict = person_data.to_dict() # in case of Series
    response = requests.post(post_url, json=person_data_dict).json()
    return response

### One request

In [5]:
case = df.sample(n=1)
result = case['is_fraud']
case_data = case.drop(columns='is_fraud')

print("Is fraud:", bool(result.iloc[0]))
response = server_request(case_data)
print("Server's response:")
print(response)

Is fraud: False
Server's response:
{'fraud_probability': 0.0024504028260707855, 'is_fraud': False}


### Multiple requests

In [6]:
N = 2000
df_sample = df.sample(n=N)
case_data = df_sample.drop(columns='is_fraud')
y_real = df_sample['is_fraud']
y_pred = pd.Series(y_real.shape)

for ind in range(N):
    y_pred[ind] = int(server_request(case_data.iloc[ind, :])['is_fraud'])
    if ind % 100 == 0: print(ind)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900


In [7]:
confusion_matrix(y_real, y_pred) / N

array([[0.962, 0.021],
       [0.004, 0.013]])